# Hospital Readmission Prediction

In [5]:
import numpy as np
import pandas as pd

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# modeling
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import (
    average_precision_score, precision_recall_curve,
    f1_score, precision_score, recall_score
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Data Loading and Exploration

In [7]:
import os

DATA_PATH = "diabetic_data.csv"
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

Shape: (101766, 50)
Columns: ['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'payer_code', 'medical_specialty', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted']


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [8]:
print("Shape:", df.shape)
display(df.dtypes)
print("\nMissing values per column:")
display(df.isna().sum())

display(df.describe(include="all").T)

Shape: (101766, 50)


,0
encounter_id,int64
patient_nbr,int64
race,object
gender,object
age,object
weight,object
admission_type_id,int64
discharge_disposition_id,int64
admission_source_id,int64
time_in_hospital,int64



Missing values per column:


,0
encounter_id,0
patient_nbr,0
race,0
gender,0
age,0
weight,0
admission_type_id,0
discharge_disposition_id,0
admission_source_id,0
time_in_hospital,0


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
encounter_id,101766.0,NaN,NaN,NaN,165201645.622978,102640295.983458,12522.0,84961194.0,152388987.0,230270887.5,443867222.0
patient_nbr,101766.0,NaN,NaN,NaN,54330400.694947,38696359.346534,135.0,23413221.0,45505143.0,87545949.75,189502619.0
race,101766,6,Caucasian,76099,NaN,NaN,NaN,NaN,NaN,NaN,NaN
gender,101766,3,Female,54708,NaN,NaN,NaN,NaN,NaN,NaN,NaN
age,101766,10,[70-80),26068,NaN,NaN,NaN,NaN,NaN,NaN,NaN
weight,101766,10,?,98569,NaN,NaN,NaN,NaN,NaN,NaN,NaN
admission_type_id,101766.0,NaN,NaN,NaN,2.024006,1.445403,1.0,1.0,1.0,3.0,8.0
discharge_disposition_id,101766.0,NaN,NaN,NaN,3.715642,5.280166,1.0,1.0,1.0,4.0,28.0
admission_source_id,101766.0,NaN,NaN,NaN,5.754437,4.064081,1.0,1.0,7.0,7.0,25.0
time_in_hospital,101766.0,NaN,NaN,NaN,4.395987,2.985108,1.0,2.0,4.0,6.0,14.0


In [9]:
TARGET = 'readmitted'
DROP_COLS = [
    "encounter_id",       # metadata
    "patient_nbr",        # identifier
    "weight",             # >96% missing
    "payer_code",         # >40% missing
    "medical_specialty",  # high missingness + many rare categories
]
CAT_COL = [
    "race", "gender", "age",
    "admission_type_id", "discharge_disposition_id", "admission_source_id",
    "diag_1", "diag_2", "diag_3",
    "max_glu_serum", "A1Cresult",
    "metformin", "repaglinide", "nateglinide", "chlorpropamide",
    "glimepiride", "acetohexamide", "glipizide", "glyburide",
    "tolbutamide", "pioglitazone", "rosiglitazone", "acarbose",
    "miglitol", "troglitazone", "tolazamide", "examide", "citoglipton",
    "insulin", "glyburide-metformin", "glipizide-metformin",
    "glimepiride-pioglitazone", "metformin-rosiglitazone",
    "metformin-pioglitazone", "change", "diabetesMed",
]
DRUG_COLS = [
    "metformin", "repaglinide", "nateglinide", "chlorpropamide",
    "glimepiride", "acetohexamide", "glipizide", "glyburide",
    "tolbutamide", "pioglitazone", "rosiglitazone", "acarbose",
    "miglitol", "troglitazone", "tolazamide", "examide", "citoglipton",
    "insulin", "glyburide-metformin", "glipizide-metformin",
    "glimepiride-pioglitazone", "metformin-rosiglitazone",
    "metformin-pioglitazone",
]
NUM_COLS = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_diagnoses",
    "number_inpatient",
    "number_outpatient",
    "number_emergency",
]
ID_AS_CAT = ["admission_type_id", "discharge_disposition_id", "admission_source_id"]
VISIT_COLS = ["number_inpatient", "number_outpatient", "number_emergency"]

# Clean + Feature Engineering

In [10]:
def make_binary_target(df: pd.DataFrame) -> pd.Series:
    """
    Binary target (unambiguous):
    Positive=1 if readmitted == "<30", else 0 for {">30","NO"}.
    """
    if TARGET not in df.columns:
        raise ValueError(f"Missing target column '{TARGET}'")
    return (df[TARGET] == "<30").astype(int)

In [11]:
def clean_df(df: pd.DataFrame,
             drop_cols=DROP_COLS,
             id_as_cat=ID_AS_CAT,
             missing_token="?") -> pd.DataFrame:
    """
    Cleaning:
    - Replace '?' with NaN
    - Drop ID/high-missing/high-cardinality columns
    - Cast id-coded categorical ints to string (so they get one-hot encoded later)
    """
    X = df.copy()

    # 1) normalize missing markers
    X = X.replace(missing_token, np.nan)

    # 2) drop specified columns if present
    drop_present = [c for c in drop_cols if c in X.columns]
    X = X.drop(columns=drop_present, errors="ignore")

    # 3) cast id-coded categoricals to string (safe for NaN)
    for c in id_as_cat:
        if c in X.columns:
            # keep NA-safe integer then to string
            X[c] = pd.to_numeric(X[c], errors="coerce").astype("Int64").astype("string")

    return X

In [12]:
def _icd_to_group(code) -> str:
    """
    Coarse ICD-9 grouping for diag_1/2/3.
    Handles:
      - V-codes, E-codes
      - numeric codes with decimals (e.g., '250.13')
    """
    if pd.isna(code):
        return np.nan
    c = str(code).strip().upper()
    if c == "" or c in {"NAN", "NONE"}:
        return np.nan

    if c.startswith("V"):
        return "supplementary_V"
    if c.startswith("E"):
        return "external_E"

    # parse numeric prefix (allow decimals)
    num_part = ""
    for ch in c:
        if ch.isdigit() or ch == ".":
            num_part += ch
        else:
            break
    try:
        val = float(num_part)
    except Exception:
        return "other"

    # common coarse bins for this dataset
    if 1 <= val < 140:   return "infectious"
    if 140 <= val < 240: return "neoplasms"
    if 240 <= val < 280: return "endocrine"
    if 280 <= val < 290: return "blood"
    if 290 <= val < 320: return "mental"
    if 320 <= val < 390: return "nervous"
    if 390 <= val < 460: return "circulatory"
    if 460 <= val < 520: return "respiratory"
    if 520 <= val < 580: return "digestive"
    if 580 <= val < 630: return "genitourinary"
    if 630 <= val < 680: return "pregnancy"
    if 680 <= val < 710: return "skin"
    if 710 <= val < 740: return "musculoskeletal"
    if 740 <= val < 760: return "congenital"
    if 760 <= val < 780: return "perinatal"
    if 780 <= val < 800: return "symptoms"
    if 800 <= val < 1000:return "injury"
    return "other"

In [13]:
def feature_engineer(df: pd.DataFrame,
                     visit_cols=VISIT_COLS,
                     drug_cols=DRUG_COLS,
                     drop_raw_diag=True,
                     drop_individual_drugs=True) -> pd.DataFrame:
    """
    Feature engineering:
    - total_visits from inpatient/outpatient/emergency
    - diag_1/2/3 -> diag_*_grp
    - drug summary features:
        num_active_drugs, insulin_flag, med_change_flag
    Optionally drops raw diag codes and/or individual drug columns.
    """
    X = df.copy()

    # total_visits
    if all(c in X.columns for c in visit_cols):
        for c in visit_cols:
            X[c] = pd.to_numeric(X[c], errors="coerce")
        X["total_visits"] = (
            X[visit_cols[0]].fillna(0) +
            X[visit_cols[1]].fillna(0) +
            X[visit_cols[2]].fillna(0)
        )

    # diagnosis grouping
    diag_cols = [c for c in ["diag_1", "diag_2", "diag_3"] if c in X.columns]
    for c in diag_cols:
        X[c + "_grp"] = X[c].map(_icd_to_group).astype("string")

    if drop_raw_diag and diag_cols:
        X = X.drop(columns=diag_cols, errors="ignore")

    # drug summary
    present_drugs = [c for c in drug_cols if c in X.columns]
    if present_drugs:
        for c in present_drugs:
            X[c] = X[c].astype("string")

        # active if not "No"
        active = pd.DataFrame({
            c: (X[c].notna() & (X[c].str.upper() != "NO")).astype(int)
            for c in present_drugs
        })
        X["num_active_drugs"] = active.sum(axis=1)

        # insulin_flag
        if "insulin" in X.columns:
            X["insulin_flag"] = (X["insulin"].notna() & (X["insulin"].str.upper() != "NO")).astype(int).astype("Int64")
        else:
            X["insulin_flag"] = 0

        # med_change_flag from "change"
        if "change" in X.columns:
            X["med_change_flag"] = (X["change"].astype("string").str.upper() == "CH").astype(int).astype("Int64")
        else:
            X["med_change_flag"] = 0

        # optional: drop individual drug columns to reduce one-hot explosion
        if drop_individual_drugs:
            keep_highlevel = {"change", "diabetesMed"}
            drop_drugs = [c for c in present_drugs if c not in keep_highlevel]
            X = X.drop(columns=drop_drugs, errors="ignore")

    return X

In [14]:
# y: binary label
y = make_binary_target(df)

# X: clean + engineered features
X = clean_df(df).drop(columns=[TARGET], errors="ignore")
X = feature_engineer(X)

print(X.shape, y.shape)
X.head()

(101766, 25) (101766,)


,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,num_lab_procedures,num_procedures,num_medications,...,A1Cresult,change,diabetesMed,total_visits,diag_1_grp,diag_2_grp,diag_3_grp,num_active_drugs,insulin_flag,med_change_flag
0,Caucasian,Female,[0-10),6,25,1,1,41,0,1,...,NaN,No,No,0,endocrine,<NA>,<NA>,0,0,0
1,Caucasian,Female,[10-20),1,1,7,3,59,0,18,...,NaN,Ch,Yes,0,endocrine,endocrine,endocrine,1,1,1
2,AfricanAmerican,Female,[20-30),1,1,7,2,11,5,13,...,NaN,No,Yes,3,pregnancy,endocrine,supplementary_V,1,0,0
3,Caucasian,Male,[30-40),1,1,7,2,44,1,16,...,NaN,Ch,Yes,0,infectious,endocrine,circulatory,1,1,1
4,Caucasian,Male,[40-50),1,1,7,1,51,0,8,...,NaN,Ch,Yes,0,neoplasms,neoplasms,endocrine,2,1,1


In [15]:
# sanity check
X.dtypes

,0
race,object
gender,object
age,object
admission_type_id,string[python]
discharge_disposition_id,string[python]
admission_source_id,string[python]
time_in_hospital,int64
num_lab_procedures,int64
num_procedures,int64
num_medications,int64


In [16]:
num_cols = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_diagnoses",
    "total_visits", # only keep this (number_outpatient + number_emergency
                    # + number_inpatient)
    "num_active_drugs",
    "insulin_flag",
    "med_change_flag",
]

cat_cols = [
    "race", "gender", "age",
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",
    "max_glu_serum",
    "A1Cresult",
    "change",
    "diabetesMed",
    "diag_1_grp",
    "diag_2_grp",
    "diag_3_grp",
]

In [17]:
# Convert all pandas 'string[python]' dtype columns to 'object' dtype (plain Python strings).
# This will turn any pd.NA into None.
for col in X.columns:
    if X[col].dtype.name == 'string':
        X[col] = X[col].astype(object)

# Replace any remaining pd.NA (e.g., from Int64 or existing object columns) with np.nan.
# This will also convert Int64 columns containing pd.NA to float64.
X = X.replace({pd.NA: np.nan})

# Explicitly ensure all columns designated as numerical are float type.
# This is for robustness and to handle columns that might have started as int64 but now have np.nan.
for c in num_cols:
    if c in X.columns:
        X[c] = pd.to_numeric(X[c], errors='coerce').astype(float)

# Preprocessing

## Train/Test Split

- 80/20 stratified split
- Holdout test set is untouched until final reporting

In [18]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Positive rate (train):", y_train.mean())

Train shape: (81412, 25)
Test shape: (20354, 25)
Positive rate (train): 0.11160516877118852


## Pipeline

In [19]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols),
])

## Baseline Model: Logistic Regression (No SMOTE)

- Use class_weight="balanced"
- CV: 5-fold Stratified
- Report PR-AUC and F1 (positive class)

In [20]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import average_precision_score

def pr_auc_scorer(estimator, X, y):
    prob = estimator.predict_proba(X)[:, 1]
    return average_precision_score(y, prob)

lr_model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    n_jobs=-1
)

pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", lr_model),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "pr_auc": pr_auc_scorer,
    "f1": "f1",
    "recall": "recall",
    "precision": "precision"
}

cv_results = cross_validate(
    pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

print("CV Results:")
for key in cv_results:
    if key.startswith("test_"):
        print(key, np.mean(cv_results[key]))

CV Results:
test_pr_auc 0.19451002298908154
test_f1 0.2619030611907711
test_recall 0.5534862346994192
test_precision 0.1715424038083916
